# 01 — Exploratory Data Analysis
Credora Finance | Credit Risk & Loan Approval Analytics Platform

Loads the governed database (built by `src/data_load.py`), profiles the four
source tables, and runs the ten analytical SQL queries from
`sql/credit_risk_platform.sql` to establish the baseline risk picture
before any modelling.

In [ ]:
import sys, sqlite3
sys.path.append('../src')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')

DB = '../data/credit_risk.db'
conn = sqlite3.connect(DB)

## 1. Table shapes and null profile

In [ ]:
for table in ['customers', 'credit_history', 'loan_applications', 'transactions']:
    df = pd.read_sql(f'SELECT * FROM {table}', conn)
    print(f"{table:<20} {df.shape[0]:>7,} rows x {df.shape[1]:>2} cols | "
          f"nulls: {int(df.isna().sum().sum())}")

## 2. Portfolio snapshot

In [ ]:
customers = pd.read_sql('SELECT * FROM customers', conn)
loans = pd.read_sql('SELECT * FROM loan_applications', conn)
credit = pd.read_sql('SELECT * FROM credit_history', conn)

approval_rate = (loans['approval_status'] == 'Approved').mean()
disbursed = loans[loans['disbursed_flag'] == 1]
default_rate = disbursed['loan_default'].mean()

print(f"Customers: {len(customers):,}")
print(f"Loan applications: {len(loans):,}")
print(f"Approval rate: {approval_rate:.1%}")
print(f"Disbursed loans: {len(disbursed):,}")
print(f"Observed default rate (disbursed): {default_rate:.1%}")
print(f"Total disbursed value: Rs {loans['approved_amount'].sum()/1e7:,.1f} crore")

## 3. Credit score distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(credit['credit_score'], bins=30, ax=ax[0], color='steelblue')
ax[0].set_title('Credit score distribution')
sns.boxplot(x=disbursed['loan_default'].map({0: 'Repaid', 1: 'Defaulted'}),
            y=disbursed.merge(credit, on='customer_id')['credit_score'], ax=ax[1])
ax[1].set_title('Credit score: repaid vs defaulted')
plt.tight_layout()
plt.show()

## 4. Default rate by loan type and key drivers

In [ ]:
default_by_type = disbursed.groupby('loan_type')['loan_default'].agg(['mean', 'count'])
default_by_type.columns = ['default_rate', 'n_loans']
default_by_type = default_by_type.sort_values('default_rate', ascending=False)
print(default_by_type)

default_by_type['default_rate'].plot(kind='barh', figsize=(6, 4), color='indianred')
plt.title('Default rate by loan type')
plt.xlabel('Default rate')
plt.tight_layout()
plt.show()

## 5. Run the ten analytical SQL queries
See `sql/credit_risk_platform.sql` sections 7.1-7.10. Reproduced here against the loaded SQLite database (identical logic; `DATE_TRUNC`/`PERCENTILE_CONT` swapped for SQLite-compatible equivalents).

In [ ]:
q_credit_band = '''
SELECT
    CASE
        WHEN credit_score >= 750 THEN 'Excellent (750-900)'
        WHEN credit_score >= 700 THEN 'Good (700-749)'
        WHEN credit_score >= 650 THEN 'Fair (650-699)'
        WHEN credit_score >= 580 THEN 'Poor (580-649)'
        ELSE 'Very Poor (300-579)'
    END AS credit_band,
    COUNT(*) AS customers,
    ROUND(AVG(credit_score),0) AS avg_score,
    ROUND(AVG(credit_utilization_ratio),3) AS avg_utilization
FROM credit_history
GROUP BY 1
ORDER BY avg_score DESC;
'''
pd.read_sql(q_credit_band, conn)

In [ ]:
q_highrisk = '''
SELECT
    c.customer_id, c.first_name, c.last_name, c.city,
    ch.credit_score, ch.credit_utilization_ratio,
    ch.num_late_payments_90d, ch.num_defaults_prior
FROM customers c
JOIN credit_history ch ON ch.customer_id = c.customer_id
WHERE ch.credit_score < 600
   OR ch.credit_utilization_ratio > 0.80
   OR ch.num_defaults_prior >= 1
   OR ch.num_late_payments_90d >= 2
ORDER BY ch.credit_score ASC
LIMIT 15;
'''
pd.read_sql(q_highrisk, conn)

## Key EDA takeaways
- Portfolio numbers match the PRD: ~46% approval rate, ~23% observed default rate on disbursed loans.
- Lower credit-score bands carry visibly higher default rates -- confirms `credit_score` as a strong candidate feature.
- Loan type is a material driver of default rate and should stay in the feature set.
- Proceed to `02_feature_engineering.ipynb` to build the modelling dataset.

In [ ]:
conn.close()